In [ ]:
import pandas as pd

# ==============================
# FILE PATHS
# ==============================
mapping_file = r"D:/Tushar/main_with_subs_only.xlsx"
comparison_file = r"D:/Tushar/Comparison data.xlsx"

# ==============================
# READ FILES
# ==============================
mapping = pd.read_excel(mapping_file)
comparison = pd.read_excel(comparison_file)

mapping.columns = mapping.columns.str.strip()
comparison.columns = comparison.columns.str.strip()

def normalize(series):
    return series.astype(str).str.strip().str.upper()

# Clean keys
mapping['Sub_Label'] = normalize(mapping['Sub_Label'])
mapping['Main_Label'] = normalize(mapping['Main_Label'])
comparison['Material'] = normalize(comparison['Material'])

mapping['Sub_Count'] = pd.to_numeric(mapping['Sub_Count'], errors='coerce').fillna(0)

# Merge
merged = mapping.merge(
    comparison,
    left_on='Sub_Label',
    right_on='Material',
    how='left'
)

# Identify columns
indent_cols = sorted([c for c in merged.columns if 'Indent' in c])
actual_cols = sorted([c for c in merged.columns if 'Production Plan' in c])

# Prepare result
child_parts = merged['Main_Label'].unique()
wide_df = pd.DataFrame({'Child_Part': child_parts})

# ==============================
# LOOP THROUGH DAYS
# ==============================
for indent_col, actual_col in zip(indent_cols, actual_cols):

    date = indent_col.split()[0]

    merged['Tentative'] = pd.to_numeric(merged[indent_col], errors='coerce').fillna(0)
    merged['Actual'] = pd.to_numeric(merged[actual_col], errors='coerce').fillna(0)

    merged['Tentative_Child'] = merged['Tentative'] * merged['Sub_Count']
    merged['Actual_Child'] = merged['Actual'] * merged['Sub_Count']

    daily = merged.groupby('Main_Label').agg({
        'Actual_Child': 'sum',
        'Tentative_Child': 'sum'
    }).reset_index()

    daily.rename(columns={
        'Main_Label': 'Child_Part',
        'Actual_Child': f"{date} Actual",
        'Tentative_Child': f"{date} Tentative"
    }, inplace=True)

    wide_df = wide_df.merge(daily, on='Child_Part', how='left')

# Fill blanks
wide_df = wide_df.fillna(0)

print(wide_df.head())

# ==============================
# SAVE
# ==============================
output_file = "Child_Comparison_Wide.xlsx"
wide_df.to_excel(output_file, index=False)

print("\nSaved to:", output_file)


In [ ]:
import pandas as pd

mapping_file = r"D:/Tushar/main_with_subs_only.xlsx"
comparison_file = r"D:/Tushar/Comparison data.xlsx"

mapping = pd.read_excel(mapping_file)
comparison = pd.read_excel(comparison_file)

mapping.columns = mapping.columns.str.strip()
comparison.columns = comparison.columns.str.strip()

def normalize(series):
    return series.astype(str).str.strip().str.upper()

mapping['Sub_Label'] = normalize(mapping['Sub_Label'])
mapping['Main_Label'] = normalize(mapping['Main_Label'])
comparison['Material'] = normalize(comparison['Material'])

mapping['Sub_Count'] = pd.to_numeric(mapping['Sub_Count'], errors='coerce').fillna(0)

merged = mapping.merge(
    comparison,
    left_on='Sub_Label',
    right_on='Material',
    how='left'
)

indent_cols = sorted([c for c in merged.columns if c.endswith('Indent')])
actual_cols = sorted([c for c in merged.columns if c.endswith('Production Plan')])

child_list = merged['Main_Label'].unique()
wide_df = pd.DataFrame({'Child_Part': child_list})

for indent_col, actual_col in zip(indent_cols, actual_cols):

    date = indent_col.split()[0]

    merged['Tentative'] = pd.to_numeric(merged[indent_col], errors='coerce').fillna(0)
    merged['Actual'] = pd.to_numeric(merged[actual_col], errors='coerce').fillna(0)

    merged['Tentative_Child'] = merged['Tentative'] * merged['Sub_Count']
    merged['Actual_Child'] = merged['Actual'] * merged['Sub_Count']

    daily = merged.groupby('Main_Label').agg({
        'Tentative_Child': 'sum',
        'Actual_Child': 'sum'
    }).reset_index()

    daily.rename(columns={
        'Main_Label': 'Child_Part',
        'Tentative_Child': f"{date} Tentative",
        'Actual_Child': f"{date} Actual"
    }, inplace=True)

    wide_df = wide_df.merge(daily, on='Child_Part', how='left')

wide_df = wide_df.fillna(0)

print(wide_df.head())

wide_df.to_excel("Child_Daily_Comparison_Wide.xlsx", index=False)

print("\nSaved: Child_Daily_Comparison_Wide.xlsx")
